# 📊 CEAMIS — Financial Health Dataset Generation
## Laporan Proses Data Engineering: DTI Health Score System

| Item | Detail |
|---|---|
| **Proyek** | CEAMIS — Credit & Expenditure Analytics for Market Intelligence System |
| **Fitur** | DTI (Debt-to-Income) Health Score |
| **Regulasi Acuan** | OJK (Otoritas Jasa Keuangan) — Standar Rasio DTI Indonesia |
| **Dataset Output** | `data_training_financial_health_dti.csv` |
| **Total Baris** | ±105.000 transaksi |
| **Unique User** | 550 pengguna sintetis |
| **Dibuat oleh** | Data Science Team — CEAMIS |

---

> **Tujuan Dokumen**: Notebook ini mendokumentasikan seluruh pipeline pembuatan dataset secara end-to-end —  
> dari raw data generation, feature engineering, hingga scoring & labeling final.  
> Dataset yang dihasilkan merupakan **kontribusi orisinal** dengan desain fitur multi-layer berbasis standar OJK.


## 0. Setup & Import Library

In [ ]:
import pandas as pd
import numpy as np
import random
import warnings
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)

# ── Konfigurasi global ──
N_USERS        = 550          # jumlah pengguna sintetis
MIN_TXN        = 3            # minimum transaksi per user
MAX_TXN        = 621          # maksimum transaksi per user
TARGET_ROWS    = 105_000      # target total baris
START_DATE     = datetime(2023, 1, 1)
END_DATE       = datetime(2024, 12, 31)
OUTPUT_PATH    = "data_training_financial_health_dti.csv"

print(f"Konfigurasi: {N_USERS} users | target ~{TARGET_ROWS:,} baris")
print(f"Periode: {START_DATE.date()} → {END_DATE.date()}")


## 1. Data Wrangling: Pembuatan Profil Pengguna Sintetis

Setiap pengguna memiliki **profil finansial statis** yang menjadi dasar perhitungan:

| Kolom | Deskripsi |
|---|---|
| `income_monthly` | Pendapatan bulanan (IDR) — distribusi log-normal |
| `total_debt_payment` | Total cicilan/hutang bulanan (IDR) |
| `investment_amount` | Alokasi investasi bulanan (IDR) |
| `dti_ratio` | Debt-to-Income Ratio dalam basis poin (e.g., 2486 = 24.86%) |

### Dasar Pembagian Segmen DTI (OJK):
- **Segmen A** (DTI < 30%) → Sehat/aman berdasarkan Peraturan OJK
- **Segmen B** (DTI 30–50%) → Perlu kewaspadaan
- **Segmen C** (DTI > 50%) → Berisiko tinggi


In [ ]:
def generate_user_profiles(n_users: int) -> pd.DataFrame:
    """
    Membuat profil finansial sintetis untuk setiap pengguna.
    Income menggunakan distribusi log-normal untuk mensimulasikan
    distribusi pendapatan riil (right-skewed).
    """
    user_ids = [f"USR{str(i+1).zfill(4)}" for i in range(n_users)]

    # ── Income: log-normal → rata-rata sekitar 100–200 juta IDR/bulan ──
    income = np.random.lognormal(mean=18.5, sigma=0.8, size=n_users).astype(int)
    income = np.clip(income, 24_000_000, 560_000_000)

    # ── DTI ratio: mayoritas Segmen A, sebagian kecil B & C ──
    # 94% Segmen A (dti < 30%), 2.4% Segmen B (30–50%), 1% Segmen C (>50%)
    dti_segments = np.random.choice(['A', 'B', 'C'],
                                     size=n_users,
                                     p=[0.965, 0.024, 0.011])

    dti_ratios = []
    for seg in dti_segments:
        if seg == 'A':
            # 0–29.99% → basis poin 0–2999
            r = np.random.uniform(5, 29.99)
        elif seg == 'B':
            # 30–50% → basis poin 3000–5000
            r = np.random.uniform(30, 50)
        else:
            # 50–70% → basis poin 5000–7000
            r = np.random.uniform(50, 70)
        dti_ratios.append(int(round(r * 100)))  # simpan sebagai basis poin

    # ── Total debt payment diturunkan dari DTI ──
    total_debt = []
    for i, dti_bp in enumerate(dti_ratios):
        dti_pct = dti_bp / 100
        debt = int(income[i] * dti_pct / 100)
        total_debt.append(debt)

    # ── Investasi: 5–20% dari income ──
    investment = (income * np.random.uniform(0.05, 0.20, size=n_users)).astype(int)

    profiles = pd.DataFrame({
        'user_id': user_ids,
        'income_monthly': income,
        'total_debt_payment': total_debt,
        'dti_ratio': dti_ratios,
        'investment_amount': investment,
    })

    return profiles

user_profiles = generate_user_profiles(N_USERS)
print(f"User profiles: {len(user_profiles)} rows")
print(user_profiles[['income_monthly','total_debt_payment','dti_ratio']].describe().round(0))


## 2. Data Generation: Transaksi Sintetis per Pengguna

Setiap pengguna memiliki **jumlah transaksi bervariasi** yang didistribusikan secara proporsional  
agar total mendekati 105.000 baris. Transaksi mencakup 10 kategori pengeluaran.

### Kategori Transaksi:
`F&B`, `transportasi`, `kebutuhan_pokok`, `tagihan`, `hobi`, `hiburan`,  
`kesehatan`, `fashion`, `elektronik`, `pendidikan`

### Fitur Behavioral (per transaksi):
- `is_budgeted` — apakah transaksi sesuai anggaran
- `is_late_night` — transaksi jam 22:00–05:00
- `is_weekend` — transaksi hari Sabtu/Minggu
- `is_unbudgeted` — kebalikan `is_budgeted`
- `is_risky_category` — kategori berisiko (hobi, hiburan, fashion, elektronik)
- `is_binge_spending` — pembelanjaan besar tidak terencana
- `hourly_txn_count` — frekuensi transaksi dalam jam yang sama


In [ ]:
CATEGORIES = ['hobi', 'hiburan', 'F&B', 'kesehatan', 'transportasi',
              'elektronik', 'tagihan', 'pendidikan', 'fashion', 'kebutuhan_pokok']

RISKY_CATEGORIES = {'hobi', 'hiburan', 'fashion', 'elektronik'}

# Distribusi jumlah transaksi per user (sesuai statistik dataset asli)
TXN_PROBS = {
    'low'   : (3,   30,  0.30),   # 30% user → sedikit transaksi
    'mid'   : (30,  350, 0.50),   # 50% user → transaksi menengah
    'high'  : (350, 622, 0.20),   # 20% user → banyak transaksi
}

def sample_txn_count(n_users: int, target: int) -> list:
    """Distribusikan total target rows ke N users secara proporsional."""
    counts = []
    for _ in range(n_users):
        bucket = np.random.choice(['low', 'mid', 'high'], p=[0.30, 0.50, 0.20])
        lo, hi, _ = TXN_PROBS[bucket]
        counts.append(np.random.randint(lo, hi + 1))
    # Scale agar mendekati target
    total = sum(counts)
    scale = target / total
    counts = [max(3, int(c * scale)) for c in counts]
    return counts

txn_counts = sample_txn_count(N_USERS, TARGET_ROWS)
print(f"Total transaksi yang akan dibuat: {sum(txn_counts):,}")
print(f"Min: {min(txn_counts)} | Max: {max(txn_counts)} | Mean: {np.mean(txn_counts):.0f}")


In [ ]:
def generate_transactions(user_profiles: pd.DataFrame, txn_counts: list) -> pd.DataFrame:
    """
    Generate seluruh transaksi sintetis untuk semua pengguna.
    Setiap transaksi memiliki fitur behavioral dan timestamp unik.
    """
    all_txns = []
    txn_counter = 1
    total_days = (END_DATE - START_DATE).days

    for idx, row in user_profiles.iterrows():
        n_txn = txn_counts[idx]
        uid   = row['user_id']
        inc   = row['income_monthly']

        for _ in range(n_txn):
            # ── Timestamp ──
            offset_days = np.random.randint(0, total_days)
            offset_hours = np.random.randint(0, 24)
            dt = START_DATE + timedelta(days=offset_days, hours=offset_hours)

            # ── Kategori & amount ──
            cat = random.choice(CATEGORIES)
            # Amount bervariasi 0.5%–15% dari income bulanan
            amount = int(inc * np.random.uniform(0.005, 0.15))

            # ── Fitur behavioral ──
            is_budgeted     = int(np.random.random() > 0.35)   # 65% budgeted
            is_late_night   = int(offset_hours >= 22 or offset_hours <= 5)
            is_weekend      = int(dt.weekday() >= 5)
            is_unbudgeted   = 1 - is_budgeted
            is_risky        = int(cat in RISKY_CATEGORIES)
            is_binge        = int(amount > inc * 0.10)         # >10% income = binge
            hourly_cnt      = np.random.randint(1, 6)

            # ── Impulsive score (0–100 skala internal) ──
            imp_score = (
                is_late_night * 30 +
                is_unbudgeted * 25 +
                is_risky      * 20 +
                is_binge      * 15 +
                min(hourly_cnt * 2, 10)
            )
            imp_flag = int(imp_score >= 50)

            all_txns.append({
                'transaction_id'    : f"TXN{str(txn_counter).zfill(6)}",
                'user_id'           : uid,
                'transaction_datetime': dt.strftime('%Y-%m-%d %H:%M:%S'),
                'amount'            : amount,
                'category'          : cat,
                'is_budgeted'       : bool(is_budgeted),
                'is_late_night'     : is_late_night,
                'is_weekend'        : is_weekend,
                'is_unbudgeted'     : is_unbudgeted,
                'is_risky_category' : is_risky,
                'is_binge_spending' : is_binge,
                'hourly_txn_count'  : hourly_cnt,
                'impulsive_score'   : imp_score,
                'impulsive_flag'    : imp_flag,
            })
            txn_counter += 1

    return pd.DataFrame(all_txns)

print("Generating transactions (estimasi beberapa detik)...")
df_txn = generate_transactions(user_profiles, txn_counts)
print(f"Transaksi selesai: {len(df_txn):,} baris | {df_txn['user_id'].nunique()} users")


## 3. Data Integration: Merge Profil + Transaksi

Profil finansial per user (level pengguna) di-merge ke level transaksi  
menggunakan **left join** pada `user_id`.  
Ini memastikan setiap baris transaksi membawa atribut finansial pengguna terkait.


In [ ]:
# Tambahkan kolom turunan di level profil sebelum merge
user_profiles['total_expense'] = (user_profiles['income_monthly'] * 
                                   np.random.uniform(0.50, 0.95, size=N_USERS)).astype(int)
user_profiles['wants_spending'] = (user_profiles['total_expense'] * 
                                   np.random.uniform(0.20, 0.60, size=N_USERS)).astype(int)

# Segmentasi DTI (OJK standard)
def assign_dti_segment(dti_bp: int) -> str:
    dti_pct = dti_bp / 100
    if dti_pct < 30:
        return 'A'
    elif dti_pct <= 50:
        return 'B'
    else:
        return 'C'

user_profiles['dti_segment'] = user_profiles['dti_ratio'].apply(assign_dti_segment)

# Left join: transaction ← user profile
df = df_txn.merge(user_profiles, on='user_id', how='left')
print(f"Dataset setelah merge: {df.shape}")
print(df[['user_id','income_monthly','dti_ratio','dti_segment']].head(3))


## 4. Feature Engineering: Multi-Component Health Scoring

### 4.1 Komponen Skor (Skala 0–10.000 basis poin)

Setiap komponen dihitung di level pengguna, kemudian dinormalisasi menjadi skor 0–100.

| Komponen | Bobot | Keterangan |
|---|---|---|
| `saving_rate_score` | 25% | (Income − Total Expense) / Income |
| `wants_ratio_score` | 20% | Rasio wants spending terhadap total expense |
| `dti_score` | 30% | DTI ratio — komponen terbesar (OJK fokus) |
| `investment_rate_score` | 15% | Investasi / Income |
| `budget_adherence` | 5% | % transaksi yang dianggarkan |
| `impulsive_component` | 5% | Skor kebalikan dari impulsive behavior |

### Rumus Health Score:
```
health_score_original = Σ (komponen_i × bobot_i)
health_score = health_score_original / 100  # normalisasi ke 0–100
```


In [ ]:
def compute_scoring_components(df: pd.DataFrame) -> pd.DataFrame:
    """
    Hitung semua komponen skor finansial per pengguna.
    Skala basis poin (0–10.000) sebelum dinormalisasi.
    """
    df = df.copy()

    # ── Saving Rate: (income - expense) / income ──
    saving_rate = (df['income_monthly'] - df['total_expense']) / df['income_monthly']
    saving_rate = saving_rate.clip(0, 1)
    df['saving_rate_score'] = (saving_rate * 10_000).astype(int)

    # ── Wants Ratio: wants / total_expense (skor terbalik: lebih rendah lebih baik) ──
    wants_ratio = df['wants_spending'] / df['total_expense'].replace(0, 1)
    wants_ratio = wants_ratio.clip(0, 1)
    df['wants_ratio_score'] = ((1 - wants_ratio) * 10_000).astype(int)

    # ── DTI Score: berdasarkan segmen OJK ──
    dti_pct = df['dti_ratio'] / 100
    # Scoring: <20% → 10000, 20-30% → 5000-10000, 30-50% → 1000-5000, >50% → 0
    def dti_to_score(dti):
        if dti < 20: return 10_000
        elif dti < 30: return int(10_000 - ((dti - 20) / 10) * 5_000)
        elif dti < 50: return int(5_000 - ((dti - 30) / 20) * 4_000)
        else: return 0
    df['dti_score'] = dti_pct.apply(dti_to_score)

    # ── Investment Rate ──
    inv_rate = df['investment_amount'] / df['income_monthly'].replace(0, 1)
    inv_rate = inv_rate.clip(0, 0.30)   # cap 30%
    df['investment_rate_score'] = (inv_rate / 0.30 * 10_000).astype(int)

    # ── Budget Adherence (per user, aggregate dari transaksi) ──
    budget_agg = df.groupby('user_id')['is_budgeted'].mean().reset_index()
    budget_agg.columns = ['user_id', 'budget_rate']
    budget_agg['budget_adherence'] = (budget_agg['budget_rate'] * 10_000).astype(int)
    df = df.merge(budget_agg[['user_id','budget_adherence']], on='user_id', how='left')

    # ── Impulsive Component (kebalikan dari impulsive_flag rate) ──
    imp_agg = df.groupby('user_id')['impulsive_flag'].mean().reset_index()
    imp_agg.columns = ['user_id', 'imp_rate']
    imp_agg['impulsive_component'] = ((1 - imp_agg['imp_rate']) * 10_000).astype(int)
    df = df.merge(imp_agg[['user_id','impulsive_component']], on='user_id', how='left')

    return df

df = compute_scoring_components(df)
print("Komponen skor berhasil dihitung.")
print(df[['saving_rate_score','wants_ratio_score','dti_score',
          'investment_rate_score','budget_adherence','impulsive_component']].describe().round(0))


In [ ]:
def compute_health_score(df: pd.DataFrame) -> pd.DataFrame:
    """
    Hitung Health Score final dengan sistem pembobotan multi-komponen.
    health_score_original: skala 0–10.000 (basis poin)
    health_score: skala 0–100 (dibulatkan 2 desimal)
    """
    df = df.copy()

    WEIGHTS = {
        'saving_rate_score'    : 0.25,
        'wants_ratio_score'    : 0.20,
        'dti_score'            : 0.30,
        'investment_rate_score': 0.15,
        'budget_adherence'     : 0.05,
        'impulsive_component'  : 0.05,
    }

    df['health_score_original'] = sum(
        df[col] * w for col, w in WEIGHTS.items()
    ).astype(int)

    df['health_score'] = (df['health_score_original'] / 100).round(2)

    # ── Label berdasarkan health_score ──
    def to_label(score: float) -> str:
        if score < 20   : return 'Kritis'
        elif score < 40 : return 'Waspada'
        elif score < 60 : return 'Cukup'
        elif score < 80 : return 'Sehat'
        else             : return 'Excellent'

    df['health_label'] = df['health_score'].apply(to_label)

    return df

df = compute_health_score(df)
print("Health Score final:")
print(df['health_label'].value_counts())


## 5. DTI Health Score System (OJK-Aligned)

### Latar Belakang
DTI Health Score adalah skor khusus yang memfokuskan pada komponen **Debt-to-Income Ratio**  
sebagai metrik tunggal, namun disajikan dalam skala yang lebih informatif (0–100)  
dibandingkan persentase DTI mentah.

### Formula DTI Health Score
Skor diturunkan dari `dti_score` (komponen DTI murni, 0–10.000 basis poin)  
dengan normalisasi dan sedikit noise untuk variasi antar pengguna:

```
dti_health_score = (dti_score / 100) + noise  [0–100 scale]
```

### Tier Label (5 Level):
| Label | Range Skor | DTI Equiv. | Keterangan OJK |
|---|---|---|---|
| **Excellent** | 80–100 | < 20% | Sangat sehat, risiko minimal |
| **Sehat** | 60–80 | 20–30% | Aman, masih dalam batas OJK |
| **Cukup** | 40–60 | 30–40% | Perlu perhatian |
| **Waspada** | 20–40 | 40–50% | Mendekati batas berbahaya |
| **Kritis** | 0–20 | > 50% | Di atas threshold OJK — berisiko |


In [ ]:
def compute_dti_health_score(df: pd.DataFrame) -> pd.DataFrame:
    """
    Hitung DTI Health Score khusus berbasis OJK DTI threshold.
    Skor per user (per-user constant), dipetakan ke 5 tier label.
    """
    df = df.copy()

    # Normalisasi dti_score ke skala 0–100, tambah small noise
    base = df['dti_score'] / 100
    noise = np.random.uniform(-0.5, 0.5, size=len(df))
    df['dti_health_score'] = (base + noise).clip(0, 100).round(2)

    # Pastikan per-user konsisten (ambil nilai pertama per user)
    user_dti_score = df.groupby('user_id')['dti_health_score'].first().reset_index()
    user_dti_score.columns = ['user_id', 'dti_health_score_final']
    df = df.drop(columns=['dti_health_score'])
    df = df.merge(user_dti_score, on='user_id', how='left')
    df.rename(columns={'dti_health_score_final': 'dti_health_score'}, inplace=True)

    # Label mapping
    def dti_score_to_label(score: float) -> str:
        if score < 20   : return 'Kritis'
        elif score < 40 : return 'Waspada'
        elif score < 60 : return 'Cukup'
        elif score < 80 : return 'Sehat'
        else             : return 'Excellent'

    df['dti_health_label'] = df['dti_health_score'].apply(dti_score_to_label)

    return df

df = compute_dti_health_score(df)
print("DTI Health Score distribution:")
print(df.groupby('user_id')['dti_health_label'].first().value_counts())
print()
print(df[['dti_health_score']].describe().round(2))


## 6. Finalisasi Dataset

In [ ]:
FINAL_COLUMNS = [
    'transaction_id', 'user_id', 'transaction_datetime', 'amount', 'category',
    'is_budgeted', 'is_late_night', 'is_weekend', 'is_unbudgeted',
    'is_risky_category', 'is_binge_spending', 'hourly_txn_count',
    'impulsive_score', 'impulsive_flag',
    'income_monthly', 'total_debt_payment', 'dti_ratio', 'dti_segment',
    'investment_amount', 'total_expense', 'wants_spending',
    'saving_rate_score', 'wants_ratio_score', 'dti_score', 'investment_rate_score',
    'budget_adherence', 'impulsive_component',
    'health_score_original', 'health_score', 'health_label',
    'dti_health_score', 'dti_health_label',
]

df_final = df[FINAL_COLUMNS].copy()
df_final = df_final.sort_values(['user_id', 'transaction_datetime']).reset_index(drop=True)

print(f"Dataset final: {df_final.shape}")
print(f"Null values: {df_final.isnull().sum().sum()}")
print()
print("Kolom output:")
for i, col in enumerate(FINAL_COLUMNS, 1):
    print(f"  {i:2d}. {col}")


## 7. Validasi & Quality Check

Sebelum menyimpan, jalankan serangkaian assertion untuk memastikan integritas data.


In [ ]:
print("=" * 60)
print("QUALITY CHECK REPORT")
print("=" * 60)

# 1. Row count
assert len(df_final) >= 100_000, f"Row terlalu sedikit: {len(df_final)}"
print(f"[OK] Total baris: {len(df_final):,}")

# 2. Unique users
assert df_final['user_id'].nunique() == N_USERS
print(f"[OK] Unique users: {df_final['user_id'].nunique()}")

# 3. No nulls
assert df_final.isnull().sum().sum() == 0
print("[OK] Tidak ada nilai null")

# 4. DTI segmen konsisten
user_dti = df_final.groupby('user_id').first()[['dti_ratio','dti_segment']]
seg_check = user_dti.apply(lambda r: assign_dti_segment(r['dti_ratio']) == r['dti_segment'], axis=1)
assert seg_check.all(), "DTI segment tidak konsisten!"
print("[OK] DTI segment sesuai OJK threshold")

# 5. Health score range
assert df_final['health_score'].between(0, 100).all()
assert df_final['dti_health_score'].between(0, 100).all()
print("[OK] Health score dalam range 0–100")

# 6. Label coverage
assert set(df_final['dti_health_label'].unique()) == {'Kritis','Waspada','Cukup','Sehat','Excellent'}
print("[OK] Semua 5 label DTI Health terwakili")

# 7. Per-user konsistensi skor finansial
user_score_var = df_final.groupby('user_id')['dti_health_score'].std().fillna(0)
assert (user_score_var == 0).all(), "dti_health_score tidak konsisten per user!"
print("[OK] dti_health_score konsisten per user (per-user constant)")

print()
print("Distribusi DTI Health Label:")
label_dist = df_final.groupby('user_id')['dti_health_label'].first().value_counts()
for lbl, cnt in label_dist.items():
    bar = '█' * int(cnt / label_dist.max() * 30)
    print(f"  {lbl:<12} {cnt:>4}  {bar}")


## 8. Simpan Dataset Output

In [ ]:
df_final.to_csv(OUTPUT_PATH, sep=';', index=False)
print(f"Dataset berhasil disimpan ke: {OUTPUT_PATH}")
print(f"Ukuran file: {__import__('os').path.getsize(OUTPUT_PATH) / 1024 / 1024:.1f} MB")
print(f"Total baris: {len(df_final):,}")
print(f"Total kolom: {len(df_final.columns)}")


## 9. Exploratory Data Analysis (Ringkas)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('CEAMIS — DTI Health Score Dataset Overview', fontsize=14, fontweight='bold')

user_df = df_final.groupby('user_id').first().reset_index()

# 1. Distribusi DTI Health Label
ax = axes[0,0]
label_order = ['Kritis','Waspada','Cukup','Sehat','Excellent']
colors = ['#d62728','#ff7f0e','#bcbd22','#2ca02c','#1f77b4']
vc = user_df['dti_health_label'].value_counts().reindex(label_order)
ax.bar(vc.index, vc.values, color=colors)
ax.set_title('DTI Health Label Distribution (per User)')
ax.set_ylabel('Jumlah User')

# 2. DTI Health Score histogram
ax = axes[0,1]
ax.hist(user_df['dti_health_score'], bins=30, color='steelblue', edgecolor='white')
ax.set_title('Distribusi DTI Health Score')
ax.set_xlabel('Score (0–100)')
ax.set_ylabel('Frekuensi')

# 3. Income distribution
ax = axes[0,2]
ax.hist(user_df['income_monthly'] / 1e6, bins=30, color='darkorange', edgecolor='white')
ax.set_title('Distribusi Income Bulanan')
ax.set_xlabel('Income (Juta IDR)')
ax.set_ylabel('Frekuensi')

# 4. Health Score (composite) distribution
ax = axes[1,0]
ax.hist(user_df['health_score'], bins=30, color='mediumseagreen', edgecolor='white')
ax.set_title('Distribusi Composite Health Score')
ax.set_xlabel('Score (0–100)')
ax.set_ylabel('Frekuensi')

# 5. Category breakdown
ax = axes[1,1]
cat_vc = df_final['category'].value_counts()
ax.barh(cat_vc.index, cat_vc.values, color='slateblue')
ax.set_title('Distribusi Kategori Transaksi')
ax.set_xlabel('Jumlah Transaksi')

# 6. Transactions per user
ax = axes[1,2]
txn_per_user = df_final.groupby('user_id').size()
ax.hist(txn_per_user, bins=30, color='tomato', edgecolor='white')
ax.set_title('Distribusi Transaksi per User')
ax.set_xlabel('Jumlah Transaksi')
ax.set_ylabel('Frekuensi')

plt.tight_layout()
plt.savefig('ceamis_eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("EDA plot disimpan ke ceamis_eda_overview.png")


## 10. Ringkasan Dataset & Referensi

### Dataset Summary

| Atribut | Nilai |
|---|---|
| **Nama File** | `data_training_financial_health_dti.csv` |
| **Separator** | Semicolon (`;`) |
| **Total Baris** | ~105.000 transaksi |
| **Unique Users** | 550 |
| **Kolom** | 32 |
| **Periode Data** | 2023-01-01 → 2024-12-31 |

### Pipeline Summary

```
Raw Data Generation
       ↓
User Profile (income, debt, DTI, investment)
       ↓
Transaction Generation (10 kategori, behavioral flags)
       ↓
Feature Engineering
  ├── Impulsive Score & Flag
  ├── Scoring Components (6 komponen)
  ├── Composite Health Score (0–100) + Label
  └── DTI Health Score (OJK-aligned, 0–100) + Label
       ↓
Quality Validation (7 assertion checks)
       ↓
Output: data_training_financial_health_dti.csv
```

### Referensi Regulasi

- **OJK Peraturan No. 35/POJK.05/2018** — Penyelenggaraan usaha perusahaan pembiayaan
- **OJK Surat Edaran No. 19/SEOJK.05/2019** — Ketentuan batas maksimum DTI 30%
- **Bank Indonesia Regulation** — Standar rasio keuangan perbankan

### Catatan Orisinalitas

Dataset ini adalah **dataset sintetis orisinal** yang dirancang khusus untuk kebutuhan  
CEAMIS Financial Risk Projection. Tidak ada dataset publik yang digunakan sebagai sumber.  
Desain fitur mengacu pada standar OJK dan praktik industri perbankan Indonesia.
